<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day09-discussion-2.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 9, Segment 2 discussion — does tanh split the difference?

The book page's vanishing-gradient table compares only two activations,
6 layers deep: **sigmoid** (gradient ratio ~3,100,000x between the
layer nearest the output and the layer nearest the input) and **ReLU**
(~130x). Sigmoid and tanh are close cousins (`tanh(x) = 2*sigmoid(2x) - 1`)
but tanh is zero-centered and has a steeper slope near 0.

**Before running anything, discuss with your group:** do you expect
tanh's vanishing-gradient ratio, at the same depth, to be closer to
sigmoid's (~3,100,000x) or closer to ReLU's (~130x)? Or somewhere
genuinely in between? Write down a guess and a one-sentence reason,
then run the cells below.

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(0)

def gradient_ratio(activation_fn, n_layers=6, width=64, n_inputs=64):
    layers = []
    for _ in range(n_layers):
        layers.append(nn.Linear(width, width))
        layers.append(activation_fn())
    net = nn.Sequential(*layers)

    x = torch.randn(1, n_inputs)
    out = net(x)
    loss = out.sum()
    loss.backward()

    linear_layers = [m for m in net if isinstance(m, nn.Linear)]
    grad_near_input = linear_layers[0].weight.grad.abs().mean().item()
    grad_near_output = linear_layers[-1].weight.grad.abs().mean().item()
    ratio = grad_near_output / grad_near_input if grad_near_input > 0 else float("inf")
    return grad_near_input, grad_near_output, ratio

for name, fn in [("Sigmoid", nn.Sigmoid), ("Tanh", nn.Tanh), ("ReLU", nn.ReLU)]:
    torch.manual_seed(0)   # same initialization for a fair comparison
    near_in, near_out, ratio = gradient_ratio(fn)
    print(f"{name:8}  near-input grad = {near_in:.2e}   near-output grad = {near_out:.2e}   ratio = {ratio:,.0f}x")

Sigmoid   near-input grad = 6.93e-06   near-output grad = 1.24e-01   ratio = 17,904x
Tanh      near-input grad = 2.52e-02   near-output grad = 8.79e-02   ratio = 3x
ReLU      near-input grad = 5.03e-03   near-output grad = 1.78e-02   ratio = 4x


**Discuss the real result against your prediction.** Tanh's derivative
peaks at 1.0 (at x=0) versus sigmoid's peak of 0.25 — tanh's gradients
should shrink less per layer than sigmoid's, but tanh still saturates
for large |x| exactly like sigmoid does, so it's not free of the
problem either. Does the measured ratio land where you expected?

**A follow-up worth trying:** rerun `gradient_ratio` with `n_layers=12`
instead of 6 for all three activations. Does the *gap between* sigmoid
and ReLU widen, narrow, or stay roughly the same proportionally? One
group presents both results.